# HLS4ML Model Synthesis Notebook

In [1]:
import hls4ml
import onnx
import torch
from torch import nn

from embedding.utils.cfg_handler import train_config, data_config
from embedding.utils.data_utils import load_data, delta_r_from_normalized
from embedding.dataloader import PFCandsDataset
from embedding.preprocs import PFPreProcessor
from embedding.models import TransformerEncoder, Projector

from pprint import pprint
device='cpu'

## Full Model

In [2]:
train_cfg = train_config('/scratch/rcruzcan/TAC-HEP-FPGA-Project/TAC-HEP-FPGA-Project/configs/train_config_hlt.yaml')
data_cfg = data_config('/scratch/rcruzcan/TAC-HEP-FPGA-Project/TAC-HEP-FPGA-Project/configs/data_config_smcocktail.yaml')
test_data_path = '/scratch/rcruzcan/TAC-HEP-FPGA-Project/TAC-HEP-FPGA-Project/data/embedding_hlt_smcocktail_test.pt'
model_path = '/scratch/rcruzcan/TAC-HEP-FPGA-Project/TAC-HEP-FPGA-Project/models/embedding_hlt_linformer_encoder_20260406_211840.pth'

In [3]:
feature_block, label_block, flags = load_data(test_data_path, map_location=device, max_events=100)
dataset = PFCandsDataset(feature_block, label_block, device=device)
dataloader = torch.utils.data.DataLoader(dataset, batch_size=1, shuffle=False, num_workers=0)

In [4]:
model_artifacts = torch.load(model_path, map_location=device)
hps = model_artifacts['train_cfg']['hyperparameters']

preproc = PFPreProcessor(norm_constants={}).to(device).eval() # {} -> no norm constants
encoder = TransformerEncoder(
    preproc.num_features,
    hps["embed_size"], 
    hps["latent_dim"], 
    num_heads=hps["num_heads"],
    num_layers=hps["num_layers"],
    linear_dim=hps["linear_dim"], 
    num_tokens=feature_block.size(1),
    batch_size=1,
    pairwise=False
).to(device).eval()
projector = Projector(hps["latent_dim"], hps["proj_dim"], hidden_dim=(hps["proj_dim"]*4)).to(device).eval()

preproc.load_state_dict(model_artifacts["preproc"], strict=False)
encoder.load_state_dict(model_artifacts["encoder"], strict=False)
projector.load_state_dict(model_artifacts["projector"], strict=False)
norm_constants = model_artifacts["norm_constants"]

In [5]:
class Model(torch.nn.Module):
    def __init__(self, encoder, projector, device):
        super().__init__()
        self.encoder = encoder
        self.projector = projector
        self.device = device
    
    def forward(self, x, mask):
        # x should be preprocessed
        mask = torch.cat(
            [
                torch.zeros(mask.size(0), 1, device=mask.device, dtype=torch.bool),
                mask.bool()
            ], dim=1
        )
        x = self.encoder(x, None, mask)
        x = self.projector(x)
        return x

In [6]:
# Model inference
model = Model(encoder, projector, device).to(device).eval()
with torch.no_grad():
    for x, mask, y in dataloader:
        x = x.to(device)
        x = preproc(x)
        mask = mask.to(device)
        y = y.to(device)
        out = model(x, mask)
        break
print(x.shape)
print(mask.shape)
print(out.shape)

torch.Size([1, 400, 14])
torch.Size([1, 400])
torch.Size([1, 6])


In [13]:
# config = hls4ml.utils.config.config_from_pytorch_model(
#     model=model,
#     input_shape=[('x', x.shape), ('mask', mask.shape)],
#     backend='Vitis'
# )
# pprint(config)
# print()

hls_model = hls4ml.converters.convert_from_pytorch_model(
    model=model,
    input_shape=[('x', x.shape), ('mask', mask.shape)],
    output_dir='hls4ml_prj',
    project_name='project',
    backend='Vitis',
    part='xcvu13p-fsga2577-2-e',
    clock_period=25,
    io_type='io_parallel'
)

Interpreting Model ...


TypeError: 'NoneType' object is not iterable

In [14]:
torch.onnx.export(
    model,
    (
        torch.randn(x.shape, device=device),
        torch.randint(0, 2, mask.shape, device=device).bool()
    ),
    './models/embedding_model.onnx',
    export_params=True,
    input_names=['x', 'mask'],
    output_names=['embeddings'],
)

onnx_model = onnx.load('./models/embedding_model.onnx')

# config = hls4ml.utils.config_from_onnx_model(onnx_model, granularity='name', backend='Vitis')
# pprint(config)
# print()

hls_model = hls4ml.converters.convert_from_onnx_model(
    model='./models/embedding_model.onnx',
    output_dir='hls4ml_prj',
    project_name='project',
    backend='Vitis',
    # hls_config=config,
    part='xcvu13p-fsga2577-2-e',
    clock_period=25,
    io_type='io_parallel'
)

W0510 17:41:46.668000 1025450 torch/onnx/_internal/exporter/_registration.py:110] torchvision is not installed. Skipping torchvision::nms
W0510 17:41:46.671000 1025450 torch/onnx/_internal/exporter/_registration.py:110] torchvision is not installed. Skipping torchvision::roi_align
W0510 17:41:46.672000 1025450 torch/onnx/_internal/exporter/_registration.py:110] torchvision is not installed. Skipping torchvision::roi_pool


[torch.onnx] Obtain model graph for `Model([...]` with `torch.export.export(..., strict=False)`...
[torch.onnx] Obtain model graph for `Model([...]` with `torch.export.export(..., strict=False)`... ✅
[torch.onnx] Run decompositions...


/usr/lib64/python3.13/copyreg.py:99: FutureWarning: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
  return cls.__new__(cls, *args)


[torch.onnx] Run decompositions... ✅
[torch.onnx] Translate the graph into ONNX...
[torch.onnx] Translate the graph into ONNX... ✅
[torch.onnx] Optimize the ONNX graph...
[torch.onnx] Optimize the ONNX graph... ✅
Interpreting Model ...
Output layers:  ['node_div_4']
Input shape: [400, 14]
Input shape: [400]
Topology:
Layer name: node_cat, layer type: Concatenate, current shape: [[1, 1], [1, 400]]
Layer name: node_MatMul_6, layer type: MatMul, current shape: [[1, 400, 14], [14, 128]]
Layer name: node_linear, layer type: Merge, current shape: [[1, 400, 128], [128]]
Layer name: node_cat_1, layer type: Concatenate, current shape: [[1, 1, 128], [1, 400, 128]]
Layer name: node_MatMul_10, layer type: MatMul, current shape: [[1, 401, 128], [128, 128]]
Layer name: node_linear_1, layer type: Merge, current shape: [[1, 401, 128], [128]]
Layer name: node_view, layer type: Reshape, current shape: [[1, 401, 128], [4]]
Layer name: node_transpose, layer type: Transpose, current shape: [[1, 401, 8, 16]

Exception: ERROR: Unsupported operation type: Expand

## Simple NN

In [2]:
class NeuralNetwork(nn.Module):
    def __init__(self, input_dim, hidden_dim, output_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, output_dim)
        )
    
    def forward(self, x):
        return self.net(x)

model = NeuralNetwork(input_dim=10, hidden_dim=16, output_dim=4).to(device).eval()
print(model)

NeuralNetwork(
  (net): Sequential(
    (0): Linear(in_features=10, out_features=16, bias=True)
    (1): ReLU()
    (2): Linear(in_features=16, out_features=4, bias=True)
  )
)


In [5]:
onnx_path = './models/simple_nn.onnx'

torch.onnx.export(
    model,
    torch.randn(10, 10, device=device),
    onnx_path,
    export_params=True,
    input_names=['x'],
    output_names=['logits'],
    dynamo=True
)

W0510 20:45:13.470000 1074514 torch/onnx/_internal/exporter/_registration.py:110] torchvision is not installed. Skipping torchvision::nms
W0510 20:45:13.474000 1074514 torch/onnx/_internal/exporter/_registration.py:110] torchvision is not installed. Skipping torchvision::roi_align
W0510 20:45:13.476000 1074514 torch/onnx/_internal/exporter/_registration.py:110] torchvision is not installed. Skipping torchvision::roi_pool


[torch.onnx] Obtain model graph for `NeuralNetwork([...]` with `torch.export.export(..., strict=False)`...
[torch.onnx] Obtain model graph for `NeuralNetwork([...]` with `torch.export.export(..., strict=False)`... ✅
[torch.onnx] Run decompositions...
[torch.onnx] Run decompositions... ✅
[torch.onnx] Translate the graph into ONNX...
[torch.onnx] Translate the graph into ONNX... ✅
[torch.onnx] Optimize the ONNX graph...
[torch.onnx] Optimize the ONNX graph... ✅


/usr/lib64/python3.13/copyreg.py:99: FutureWarning: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
  return cls.__new__(cls, *args)


ONNXProgram(
    model=
        <
            ir_version=10,
            opset_imports={'': 20},
            producer_name='pytorch',
            producer_version='2.11.0+cu130',
            domain=None,
            model_version=None,
        >
        graph(
            name=main_graph,
            inputs=(
                %"x"<FLOAT,[10,10]>
            ),
            outputs=(
                %"logits"<FLOAT,[10,4]>
            ),
            initializers=(
                %"net.0.weight"<FLOAT,[16,10]>{TorchTensor(...)},
                %"net.0.bias"<FLOAT,[16]>{TorchTensor(...)},
                %"net.2.weight"<FLOAT,[4,16]>{TorchTensor(...)},
                %"net.2.bias"<FLOAT,[4]>{TorchTensor<FLOAT,[4]>(Parameter containing: tensor([-0.0881, -0.0483, -0.2067,  0.0999], requires_grad=True), name='net.2.bias')}
            ),
        ) {
            0 |  # node_linear
                 %"linear"<FLOAT,[10,16]> ⬅️ ::Gemm(%"x", %"net.0.weight"{...}, %"net.0.bias"{...}) {beta=1.0, t

In [6]:
onnx_model = onnx.load(onnx_path)
onnx.checker.check_model(onnx_model)

config = hls4ml.utils.config_from_onnx_model(
    onnx_model, 
    granularity='name', 
    backend='Vitis',
    # default_precision='ap_fixed<16,6>',
    # default_reuse_factor=1,
)

pprint(config)

Output layers:  ['node_linear_1']
Input shape: [10]
Topology:


Exception: ERROR: Unsupported operation type: Gemm

---

## Other attempts

## No Mask

In [ ]:
from typing import Union
import torch
import torch.nn as nn
import torch.nn.functional as F

class LinearAttentionLayer(nn.Module):
    def __init__(self, embed_dim: int, num_heads: int, linear_dim: int, num_tokens: int):
        super().__init__()
        self.embed_dim = embed_dim
        self.num_heads = num_heads
        self.head_dim = embed_dim // num_heads
        assert embed_dim % num_heads == 0

        self.q_proj = nn.Linear(embed_dim, embed_dim)
        self.k_proj = nn.Linear(embed_dim, embed_dim)
        self.v_proj = nn.Linear(embed_dim, embed_dim)
        self.out_proj = nn.Linear(embed_dim, embed_dim)

        # Linformer projection matrices
        self.f_proj = nn.Linear(num_tokens, linear_dim, bias=False)
        self.e_proj = nn.Linear(num_tokens, linear_dim, bias=False)

        self.register_buffer('scale', torch.tensor(self.head_dim).float().sqrt())

        self.bias_mlp = nn.Sequential(
            nn.Linear(1, 16),
            nn.ReLU(),
            nn.Linear(16, num_heads)
        )

    def forward(self, x: torch.Tensor):
        B, N, E = x.shape

        Q = self.q_proj(x).view(B, N, self.num_heads, self.head_dim).transpose(1, 2)  # B,H,N,head_dim
        K = self.k_proj(x).view(B, N, self.num_heads, self.head_dim).transpose(1, 2)  # B,H,N,head_dim
        V = self.v_proj(x).view(B, N, self.num_heads, self.head_dim).transpose(1, 2)  # B,H,N,head_dim

        K_prime = self.e_proj(K.transpose(2, 3)).transpose(2, 3) # B,H,linear_dim,head_dim
        V_prime = self.f_proj(V.transpose(2, 3)).transpose(2, 3) # B,H,linear_dim,head_dim
        
        # scores = torch.matmul(Q, K_prime.transpose(-2, -1)) / torch.sqrt(torch.tensor(self.head_dim))  # (B,H,N,head_dim)x(B,H,head_dim,linear_dim) => B,H,N,linear_dim
        scores = torch.matmul(Q, K_prime.transpose(-2, -1)) / self.scale  # (B,H,N,head_dim)x(B,H,head_dim,linear_dim) => B,H,N,linear_dim

        attn = torch.softmax(scores, dim=-1)  # B,H,N,linear_dim
        out = torch.matmul(attn, V_prime)  # (B,H,N,linear_dim)x(B,H,linear_dim,head_dim) => B,H,N,head_dim

        out = out.transpose(1, 2).contiguous().view(B, N, E)
        out = self.out_proj(out)
        return out

class TransformerEncoderBlock(nn.Module):
    def __init__(
            self, 
            embed_dim: int, 
            num_heads: int, 
            dim_feedforward: int = 2048, 
            dropout: float = 0.1, 
            linear_dim: Union[int, None] = None, 
            num_tokens: Union[int, None] = None,
        ):
        super().__init__()
        if linear_dim is not None and num_tokens is None:
            raise ValueError("num_tokens must be provided if linear_dim is specified")
        self.self_attn = LinearAttentionLayer(embed_dim, num_heads, linear_dim, num_tokens)
        self.linear1 = nn.Linear(embed_dim, dim_feedforward)
        self.dropout = nn.Dropout(dropout)
        self.linear2 = nn.Linear(dim_feedforward, embed_dim)

        self.norm1 = nn.LayerNorm(embed_dim)
        self.norm2 = nn.LayerNorm(embed_dim)
        self.dropout1 = nn.Dropout(dropout)
        self.dropout2 = nn.Dropout(dropout)

        self.activation = nn.ReLU()

    def forward(
            self, 
            src: torch.Tensor, 
        ):
        src2 = self.self_attn(src)
        src = src + self.dropout1(src2)
        src = self.norm1(src)
        src2 = self.linear2(self.dropout(self.activation(self.linear1(src))))
        src = src + self.dropout2(src2)
        src = self.norm2(src)
        return src

class TransformerEncoder(nn.Module):
    def __init__(
            self, 
            num_features: int, 
            embed_size: int, 
            latent_dim: int, 
            num_heads: int = 8, 
            num_layers: int = 4,
            linear_dim: Union[int, None] = None,
            num_tokens: Union[int, None] = None,
        ):
        super().__init__()
        self.input_proj = nn.Linear(num_features, embed_size)
        self.layers = nn.ModuleList(
            [
                TransformerEncoderBlock(
                    embed_size, 
                    num_heads, 
                    linear_dim=linear_dim, 
                    num_tokens=num_tokens+1 if num_tokens is not None else None
                ) for _ in range(num_layers)
            ]
        )
        self.norm_cls_embedding = nn.LayerNorm(embed_size)
        self.cls_token = nn.Parameter(torch.randn(1, 1, embed_size))
        self.bottleneck = nn.Linear(embed_size, latent_dim)

    def forward(self, x: torch.Tensor):
        B, N, F = x.shape
        x = self.input_proj(x) # [B, N, E]

        cls_tokens = self.cls_token.expand(B, -1, -1)
        x = torch.cat([cls_tokens, x], dim=1) 
            
        for layer in self.layers:
            x = layer(x)

        cls_embedding = x[:, 0, :] # CLS token embedding
        latent = self.bottleneck(self.norm_cls_embedding(cls_embedding))
        return latent
    
class Projector(nn.Module):
    def __init__(self, input_dim, proj_dim, hidden_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.BatchNorm1d(hidden_dim),
            nn.GELU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.BatchNorm1d(hidden_dim),
            nn.GELU(),
            nn.Linear(hidden_dim, proj_dim)
        )

    def forward(self, z):
        z = self.net(z)
        norm = torch.sqrt(torch.sum(z**2, dim=-1, keepdim=True) + 1e-6)
        return z / norm

class Model(torch.nn.Module):
    def __init__(self, encoder, projector):
        super().__init__()
        self.encoder = encoder
        self.projector = projector
    
    def forward(self, x):
        x = self.encoder(x)
        x = self.projector(x)
        return x

In [9]:
feature_block, label_block, flags = load_data(test_data_path, map_location=device, max_events=1000)
dataset = PFCandsDataset(feature_block, label_block, device=device)
dataloader = torch.utils.data.DataLoader(dataset, batch_size=1000, shuffle=False, num_workers=0)

In [10]:
model_artifacts = torch.load(model_path, map_location=device)
hps = model_artifacts['train_cfg']['hyperparameters']

preproc = PFPreProcessor(norm_constants={}).to(device).eval() # {} -> no norm constants
encoder = TransformerEncoder(
    preproc.num_features,
    hps["embed_size"], 
    hps["latent_dim"], 
    num_heads=hps["num_heads"],
    num_layers=hps["num_layers"],
    linear_dim=hps["linear_dim"], 
    num_tokens=feature_block.size(1),
).to(device).eval()
projector = Projector(hps["latent_dim"], hps["proj_dim"], hidden_dim=(hps["proj_dim"]*4)).to(device).eval()

preproc.load_state_dict(model_artifacts["preproc"], strict=False)
encoder.load_state_dict(model_artifacts["encoder"], strict=False)
projector.load_state_dict(model_artifacts["projector"], strict=False)

<All keys matched successfully>

In [12]:
# Model inference
model = Model(encoder, projector).to(device).eval()
with torch.no_grad():
    for x, mask, y in dataloader:
        x = preproc(x)
        out = model(x)
        break
print(x.shape)
print(mask.shape)
print(out.shape)

torch.Size([1000, 400, 14])
torch.Size([1000, 400])
torch.Size([1000, 6])


In [ ]:
torch.onnx.export(
    preproc,
    (
        torch.randn(x.shape),
    ),
    'embedding_model.onnx',
    export_params=True,
    input_names=['x'],
    output_names=['embeddings'],
    dynamic_axes={
        'x': {0: 'batch_size'},
        'embeddings': {0: 'batch_size'}
    },
    # dynamo=True
)

config = hls4ml.utils.config_from_onnx_model('embedding_model.onnx', granularity='name')
# config['Model']['Precision'] = 'ap_fixed<16,6>'
# config['Model']['ReuseFactor'] = 1
# config['Model']['Strategy'] = 'Latency'
pprint(config)
print()

hls_model = hls4ml.converters.convert_from_onnx_model(
    model='embedding_model.onnx',
    output_dir='hls4ml_prj',
    project_name='project',
    backend='Vitis',
    hls_config=config,
    part='xcvu13p-fsga2577-2-e',
    clock_period=5,
    io_type='io_parallel'
)

/tmp/ipykernel_3309412/308351286.py:1: DeprecationWarning: You are using the legacy TorchScript-based ONNX export. Starting in PyTorch 2.9, the new torch.export-based ONNX exporter will be the default. To switch now, set dynamo=True in torch.onnx.export. This new exporter supports features like exporting LLMs with DynamicCache. We encourage you to try it and share feedback to help improve the experience. Learn more about the new export logic: https://pytorch.org/docs/stable/onnx_dynamo.html. For exporting control flow: https://pytorch.org/tutorials/beginner/onnx/export_control_flow_model_to_onnx_tutorial.html.
  torch.onnx.export(
/scratch/rcruzcan/TAC-HEP-FPGA-Project/venv/lib64/python3.9/site-packages/torch/onnx/symbolic_opset10.py:517: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  return g.op("Constant", value_t=torch.tensor(list_or_

{'Model': {'Precision': 'ap_fixed<16,6>',
           'ReuseFactor': 1,
           'Strategy': 'Latency'}}

Interpreting Model ...
Output layers:  ['/ScatterND_3']
Input shape: [None, 400, 14]
Topology:


Exception: ERROR: Unsupported operation type: Constant

```
V pass_manager.cc:41 run]: Pass eliminate_nop_cast transformed 5
[V pass_manager.cc:41 run]: Pass extract_constant_to_initializer transformed 83
[V pass_manager.cc:41 run]: Pass eliminate_nop_concat transformed 1
[V pass_manager.cc:41 run]: Pass eliminate_shape_gather transformed 4
[V pass_manager.cc:41 run]: Pass eliminate_slice_after_shape transformed 1
[V pass_manager.cc:41 run]: Pass eliminate_nop_reshape transformed 2
[V pass_manager.cc:41 run]: Pass eliminate_nop_with_unit transformed 1
[V eliminate_common_subexpression.h:52 EliminateCommonSubexpressions]: kind: NonZero, /NonZero_2 [0] output has been replaced by /NonZero
[V eliminate_common_subexpression.h:52 EliminateCommonSubexpressions]: kind: Transpose, /Transpose_2 [0] output has been replaced by /Transpose
[V eliminate_common_subexpression.h:52 EliminateCommonSubexpressions]: kind: Shape, /Shape_15 [0] output has been replaced by /Shape_14
[V eliminate_common_subexpression.h:52 EliminateCommonSubexpressions]: kind: ConstantOfShape, /ConstantOfShape_4 [0] output has been replaced by /ConstantOfShape_3
[V eliminate_common_subexpression.h:52 EliminateCommonSubexpressions]: kind: Shape, /Shape_16 [0] output has been replaced by /Shape_14
[V eliminate_common_subexpression.h:52 EliminateCommonSubexpressions]: kind: ConstantOfShape, /ConstantOfShape_5 [0] output has been replaced by /ConstantOfShape_3
[V eliminate_common_subexpression.h:64 runPass]: cse_removed count: 6
[V pass_manager.cc:41 run]: Pass eliminate_common_subexpression transformed 6
[V pass_manager.cc:41 run]: Pass eliminate_deadend transformed 12
[V pass_manager.cc:41 run]: Pass eliminate_shape_op transformed 1
[V eliminate_duplicate_initializer.h:110 EliminateInitializer]: ====== Graph: main_graph=====
[V eliminate_duplicate_initializer.h:112 EliminateInitializer]: </Constant_10_output_0,/Constant_1_output_0>
[V eliminate_duplicate_initializer.h:112 EliminateInitializer]: </Constant_12_output_0,/Constant_11_output_0>
[V eliminate_duplicate_initializer.h:112 EliminateInitializer]: </Constant_13_output_0,/Constant_11_output_0>
[V eliminate_duplicate_initializer.h:112 EliminateInitializer]: </Constant_15_output_0,/Constant_1_output_0>
[V eliminate_duplicate_initializer.h:112 EliminateInitializer]: </Constant_16_output_0,/Constant_11_output_0>
[V eliminate_duplicate_initializer.h:112 EliminateInitializer]: </Constant_17_output_0,/Constant_11_output_0>
[V eliminate_duplicate_initializer.h:112 EliminateInitializer]: </Constant_18_output_0,/Constant_11_output_0>
[V eliminate_duplicate_initializer.h:112 EliminateInitializer]: </Constant_27_output_0,onnx::ReduceSum_26>
[V eliminate_duplicate_initializer.h:112 EliminateInitializer]: </Constant_28_output_0,onnx::ReduceSum_26>
[V eliminate_duplicate_initializer.h:112 EliminateInitializer]: </Constant_29_output_0,onnx::ReduceSum_26>
[V eliminate_duplicate_initializer.h:112 EliminateInitializer]: </Constant_30_output_0,onnx::ReduceSum_26>
[V eliminate_duplicate_initializer.h:112 EliminateInitializer]: </Constant_31_output_0,onnx::ReduceSum_26>
[V eliminate_duplicate_initializer.h:112 EliminateInitializer]: </Constant_32_output_0,onnx::ReduceSum_26>
[V eliminate_duplicate_initializer.h:112 EliminateInitializer]: </Constant_33_output_0,onnx::ReduceSum_26>
[V eliminate_duplicate_initializer.h:112 EliminateInitializer]: </Constant_34_output_0,onnx::ReduceSum_26>
[V eliminate_duplicate_initializer.h:112 EliminateInitializer]: </Constant_36_output_0,/Constant_11_output_0>
[V eliminate_duplicate_initializer.h:112 EliminateInitializer]: </Constant_39_output_0,/Constant_1_output_0>
[V eliminate_duplicate_initializer.h:112 EliminateInitializer]: <onnx::Unsqueeze_138,/Constant_11_output_0>
[V eliminate_duplicate_initializer.h:112 EliminateInitializer]: <onnx::Unsqueeze_140,/Constant_11_output_0>
[V eliminate_duplicate_initializer.h:112 EliminateInitializer]: <onnx::Unsqueeze_144,/Constant_11_output_0>
[V eliminate_duplicate_initializer.h:112 EliminateInitializer]: </Constant_42_output_0,/Constant_38_output_0>
[V eliminate_duplicate_initializer.h:112 EliminateInitializer]: </Constant_43_output_0,onnx::ReduceSum_26>
[V eliminate_duplicate_initializer.h:112 EliminateInitializer]: </Constant_44_output_0,/Constant_1_output_0>
[V eliminate_duplicate_initializer.h:112 EliminateInitializer]: </Constant_45_output_0,/Constant_11_output_0>
[V eliminate_duplicate_initializer.h:112 EliminateInitializer]: </Constant_46_output_0,/Constant_11_output_0>
[V eliminate_duplicate_initializer.h:112 EliminateInitializer]: </Constant_47_output_0,/Constant_11_output_0>
[V eliminate_duplicate_initializer.h:112 EliminateInitializer]: <onnx::Unsqueeze_169,/Constant_11_output_0>
[V eliminate_duplicate_initializer.h:112 EliminateInitializer]: <onnx::Unsqueeze_171,/Constant_11_output_0>
[V eliminate_duplicate_initializer.h:112 EliminateInitializer]: <onnx::Unsqueeze_173,/Constant_11_output_0>
[V eliminate_duplicate_initializer.h:112 EliminateInitializer]: </Constant_48_output_0,/Constant_35_output_0>
[V eliminate_duplicate_initializer.h:112 EliminateInitializer]: </Constant_49_output_0,/Constant_11_output_0>
[V eliminate_duplicate_initializer.h:112 EliminateInitializer]: </Constant_50_output_0,/Constant_37_output_0>
[V eliminate_duplicate_initializer.h:112 EliminateInitializer]: </Constant_51_output_0,/Constant_38_output_0>
[V eliminate_duplicate_initializer.h:112 EliminateInitializer]: <onnx::Gather_189,/Constant_1_output_0>
[V eliminate_duplicate_initializer.h:112 EliminateInitializer]: <onnx::Range_192,/Constant_1_output_0>
[V eliminate_duplicate_initializer.h:112 EliminateInitializer]: <onnx::Range_193,/Constant_2_output_0>
[V eliminate_duplicate_initializer.h:112 EliminateInitializer]: <onnx::Range_199,/Constant_1_output_0>
[V eliminate_duplicate_initializer.h:112 EliminateInitializer]: <onnx::Range_200,/Constant_2_output_0>
[V eliminate_duplicate_initializer.h:112 EliminateInitializer]: <onnx::Range_206,/Constant_1_output_0>
[V eliminate_duplicate_initializer.h:112 EliminateInitializer]: <onnx::Range_207,/Constant_2_output_0>
[V eliminate_duplicate_initializer.h:112 EliminateInitializer]: </Constant_52_output_0,/Constant_11_output_0>
[V eliminate_duplicate_initializer.h:112 EliminateInitializer]: </Constant_53_output_0,/Constant_11_output_0>
[V eliminate_duplicate_initializer.h:112 EliminateInitializer]: </Constant_54_output_0,/Constant_37_output_0>
[V eliminate_duplicate_initializer.h:112 EliminateInitializer]: </Constant_55_output_0,/Constant_38_output_0>
[V eliminate_duplicate_initializer.h:112 EliminateInitializer]: </Constant_57_output_0,onnx::ReduceSum_26>
[V eliminate_duplicate_initializer.h:112 EliminateInitializer]: </Constant_58_output_0,/Constant_56_output_0>
[V eliminate_duplicate_initializer.h:112 EliminateInitializer]: </Constant_59_output_0,onnx::ReduceSum_26>
[V eliminate_duplicate_initializer.h:112 EliminateInitializer]: </Constant_60_output_0,/Constant_56_output_0>
[V eliminate_duplicate_initializer.h:112 EliminateInitializer]: </Constant_61_output_0,onnx::ReduceSum_26>
[V eliminate_duplicate_initializer.h:112 EliminateInitializer]: <_v_418,/Constant_5_output_0>
[V eliminate_duplicate_initializer.h:112 EliminateInitializer]: <_v_420,_v_416>
[V eliminate_duplicate_initializer.h:122 EliminateInitializer]: ====== Graph: main_graph=====, removed: 51
[V pass_manager.cc:41 run]: Pass eliminate_duplicate_initializer transformed 51
[V pass_manager.cc:41 run]: Pass eliminate_nop_concat transformed 1
[V eliminate_common_subexpression.h:52 EliminateCommonSubexpressions]: kind: Unsqueeze, /Unsqueeze_12 [0] output has been replaced by /Unsqueeze_10
[V eliminate_common_subexpression.h:52 EliminateCommonSubexpressions]: kind: Unsqueeze, /Unsqueeze_17 [0] output has been replaced by /Unsqueeze_11
[V eliminate_common_subexpression.h:52 EliminateCommonSubexpressions]: kind: Slice, /Slice_4 [0] output has been replaced by /Slice_2
[V eliminate_common_subexpression.h:52 EliminateCommonSubexpressions]: kind: Shape, /Shape_12 [0] output has been replaced by /Shape_7
[V eliminate_common_subexpression.h:52 EliminateCommonSubexpressions]: kind: Mul, /Mul_2 [0] output has been replaced by /Mul_1
[V eliminate_common_subexpression.h:52 EliminateCommonSubexpressions]: kind: Equal, /Equal_8 [0] output has been replaced by /Equal_7
[V eliminate_common_subexpression.h:52 EliminateCommonSubexpressions]: kind: Where, /Where_5 [0] output has been replaced by /Where_4
[V eliminate_common_subexpression.h:52 EliminateCommonSubexpressions]: kind: Mul, /Mul_3 [0] output has been replaced by /Mul_1
[V eliminate_common_subexpression.h:52 EliminateCommonSubexpressions]: kind: Equal, /Equal_9 [0] output has been replaced by /Equal_7
[V eliminate_common_subexpression.h:52 EliminateCommonSubexpressions]: kind: Where, /Where_6 [0] output has been replaced by /Where_4
[V eliminate_common_subexpression.h:64 runPass]: cse_removed count: 10
[V pass_manager.cc:41 run]: Pass eliminate_common_subexpression transformed 10
[V pass_manager.cc:41 run]: Pass eliminate_deadend transformed 11
[V onnxsim.cpp:294 RunOp]: Running node:    ["/Unsqueeze_11"] "/Unsqueeze_11_output_0" = Unsqueeze ("/Constant_5_output_0", "/Constant_11_output_0")

WARNING: failed to run "Unsqueeze" op (name is "/Unsqueeze_11"), skip...
[V onnxsim.cpp:294 RunOp]: Running node:    ["/Unsqueeze_16"] "/Unsqueeze_16_output_0" = Unsqueeze (_v_416, "/Constant_11_output_0")

WARNING: failed to run "Unsqueeze" op (name is "/Unsqueeze_16"), skip...
[V onnxsim.cpp:294 RunOp]: Running node:    [Range_269] "onnx::Reshape_201" = Range ("/Constant_1_output_0", _v_416, "/Constant_2_output_0")

WARNING: failed to run "Range" op (name is "Range_269"), skip...
[V onnxsim.cpp:294 RunOp]: Running node:    [Range_276] "onnx::Slice_208" = Range ("/Constant_1_output_0", _v_422, "/Constant_2_output_0")

WARNING: failed to run "Range" op (name is "Range_276"), skip...
WARNING: failed to run "Slice" op (name is "/Slice_5"), skip...
WARNING: failed to run "Reshape" op (name is "Reshape_285"), skip...
[V onnxsim.cpp:294 RunOp]: Running node:    ["/ConstantOfShape_3"] "/ConstantOfShape_3_output_0" = ConstantOfShape <value: tensor = int64[1] {1}> (_v_426)

WARNING: failed to run "ConstantOfShape" op (name is "/ConstantOfShape_3"), skip...
WARNING: failed to run "Mul" op (name is "/Mul_1"), skip...
[V eliminate_common_subexpression.h:64 runPass]: cse_removed count: 0
[V pass_manager.cc:41 run]: Pass eliminate_deadend transformed 1
[V onnxsim.cpp:294 RunOp]: Running node:    ["/Unsqueeze_11"] "/Unsqueeze_11_output_0" = Unsqueeze ("/Constant_5_output_0", "/Constant_11_output_0")

WARNING: failed to run "Unsqueeze" op (name is "/Unsqueeze_11"), skip...
[V onnxsim.cpp:294 RunOp]: Running node:    ["/Unsqueeze_16"] "/Unsqueeze_16_output_0" = Unsqueeze (_v_416, "/Constant_11_output_0")

WARNING: failed to run "Unsqueeze" op (name is "/Unsqueeze_16"), skip...
[V onnxsim.cpp:294 RunOp]: Running node:    [Range_269] "onnx::Reshape_201" = Range ("/Constant_1_output_0", _v_416, "/Constant_2_output_0")

WARNING: failed to run "Range" op (name is "Range_269"), skip...
[V onnxsim.cpp:294 RunOp]: Running node:    [Range_276] "onnx::Slice_208" = Range ("/Constant_1_output_0", _v_422, "/Constant_2_output_0")

WARNING: failed to run "Range" op (name is "Range_276"), skip...
WARNING: failed to run "Slice" op (name is "/Slice_5"), skip...
WARNING: failed to run "Reshape" op (name is "Reshape_285"), skip...
[V onnxsim.cpp:294 RunOp]: Running node:    ["/ConstantOfShape_3"] "/ConstantOfShape_3_output_0" = ConstantOfShape <value: tensor = int64[1] {1}> (_v_426)

WARNING: failed to run "ConstantOfShape" op (name is "/ConstantOfShape_3"), skip...
WARNING: failed to run "Mul" op (name is "/Mul_1"), skip...
simplify error: Nodes in a graph must be topologically sorted, however input '/Unsqueeze_11_output_0' of node: 
name: /Concat_1 OpType: Concat
 is not output of any previous nodes.
simplify failed!
```

## Simplified

In [26]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class EncoderBlock(nn.Module):
    def __init__(self, embed_dim, ff_dim, num_tokens, batch_size=1):
        super().__init__()
        self.embed_dim = embed_dim
        self.num_tokens = num_tokens
        self.batch_size = batch_size
        
        # Explicit Projections (Replacing MultiheadAttention)
        self.q_proj = nn.Linear(embed_dim, embed_dim)
        self.k_proj = nn.Linear(embed_dim, embed_dim)
        self.v_proj = nn.Linear(embed_dim, embed_dim)
        self.out_proj = nn.Linear(embed_dim, embed_dim)
        
        # Scaling factor for attention
        self.register_buffer('scale', torch.tensor(embed_dim).float().sqrt())
        
        self.norm1 = nn.BatchNorm1d(embed_dim, eps=1e-5, momentum=0.1)
        
        self.mlp = nn.Sequential(
            nn.Linear(embed_dim, ff_dim),
            nn.ReLU(),
            nn.Linear(ff_dim, embed_dim)
        )
        self.norm2 = nn.BatchNorm1d(embed_dim, eps=1e-5, momentum=0.1)

    def forward(self, x):
        # 1. Manual Attention Calculation
        # Input x shape: [B, N, E]
        q = self.q_proj(x)
        k = self.k_proj(x)
        v = self.v_proj(x)
        
        # Attention Scores: (B, N, E) x (B, E, N) -> (B, N, N)
        # Using .transpose(-2, -1) is safe for hls4ml as long as dims are static
        scores = torch.matmul(q, k.transpose(-2, -1)) / self.scale
        attn = torch.softmax(scores, dim=-1)
        
        # Context Vector: (B, N, N) x (B, N, E) -> (B, N, E)
        context = torch.matmul(attn, v)
        attn_out = self.out_proj(context)
        
        # 2. Residual + Norm
        x = x + attn_out
        B, N, E = self.batch_size, self.num_tokens, self.embed_dim
        x = self.norm1(x.reshape(B*N, E)).reshape(B, N, E)
        
        # 3. MLP + Residual + Norm
        x = x + self.mlp(x)
        x = self.norm2(x.reshape(B*N, E)).reshape(B, N, E)
        return x

class TransformerEncoder(nn.Module):
    def __init__(self, num_features, embed_dim, ff_dim, proj_dim, num_tokens, batch_size=1):
        super().__init__()
        self.batch_size = batch_size
        self.num_tokens = num_tokens
        
        self.input_proj = nn.Linear(num_features, embed_dim)
        
        # Pass static sizes down
        self.encoder = EncoderBlock(embed_dim, ff_dim, num_tokens, batch_size)
        
        self.projector = nn.Sequential(
            nn.Linear(embed_dim, proj_dim),
            nn.ReLU()
        )

    def forward(self, x):
        # x: [batch_size, num_tokens, num_features]
        x = self.input_proj(x)
        x = self.encoder(x)
        
        # Global Average Pooling (Statically defined)
        # torch.mean is fine, but we explicitly keep it on the token dimension
        x = torch.mean(x, dim=1) 
        
        x = self.projector(x)
        return x

In [29]:
encoder = TransformerEncoder(
    num_features=10,
    embed_dim=32,
    ff_dim=64,
    proj_dim=16,
    num_tokens=10,
    batch_size=100
).to(device).eval()

# Export to onnx
torch.onnx.export(
    encoder,
    torch.randn(100, 10, 10, device=device),  # [batch_size, num_tokens, num_features]
    './models/embedding_model.onnx',
    export_params=True,
    input_names=['x'],
    output_names=['embeddings'],
    dynamic_axes={
        'x': {0: 'batch_size'},
        'embeddings': {0: 'batch_size'}
    },
    dynamo=False
)

/tmp/ipykernel_918602/1951034483.py:11: DeprecationWarning: You are using the legacy TorchScript-based ONNX export. Starting in PyTorch 2.9, the new torch.export-based ONNX exporter has become the default. Learn more about the new export logic: https://docs.pytorch.org/docs/stable/onnx_export.html. For exporting control flow: https://pytorch.org/tutorials/beginner/onnx/export_control_flow_model_to_onnx_tutorial.html
  torch.onnx.export(


In [30]:
# Synthesize with hls4ml
onnx_model = onnx.load('./models/embedding_model.onnx')
config = hls4ml.utils.config_from_onnx_model(onnx_model, granularity='name', backend='Vitis')
pprint(config)
print()

hls_model = hls4ml.converters.convert_from_onnx_model(
    model='./models/embedding_model.onnx',
    output_dir='hls4ml_prj',
    project_name='project',
    backend='Vitis',
    hls_config=config,
    part='xcvu13p-fsga2577-2-e',
    clock_period=5,
    io_type='io_parallel'
)

Output layers:  ['/projector/projector.1/Relu']
Input shape: [10, 10]
Topology:


RuntimeError: Could not find the shape for input encoder.norm1.weight

In [33]:
batch_size = 1
num_tokens = 400
num_features = 14
embed_dim = 128
ff_dim = 128

device = torch.device('cpu')
model = HLSTransformer(
    num_features=num_features,
    embed_dim=embed_dim,
    ff_dim=ff_dim,
    proj_dim=embed_dim,
    num_tokens=num_tokens
).to(device).eval()

dummy_input = torch.randn(batch_size, num_tokens, num_features)

onnx_path = "model.onnx"
torch.onnx.export(
    model, 
    dummy_input, 
    onnx_path,
    export_params=True,
    input_names=['input'],
    output_names=['output']
)

/tmp/ipykernel_3309412/1051262333.py:19: DeprecationWarning: You are using the legacy TorchScript-based ONNX export. Starting in PyTorch 2.9, the new torch.export-based ONNX exporter will be the default. To switch now, set dynamo=True in torch.onnx.export. This new exporter supports features like exporting LLMs with DynamicCache. We encourage you to try it and share feedback to help improve the experience. Learn more about the new export logic: https://pytorch.org/docs/stable/onnx_dynamo.html. For exporting control flow: https://pytorch.org/tutorials/beginner/onnx/export_control_flow_model_to_onnx_tutorial.html.
  torch.onnx.export(


In [37]:
config = hls4ml.utils.config_from_onnx_model('model.simplify.onnx', granularity='name')
# config['Model']['Precision'] = 'ap_fixed<16,6>'
# config['Model']['ReuseFactor'] = 4
# config['Model']['Strategy'] = 'Latency'
pprint(config)
print()

hls_model = hls4ml.converters.convert_from_onnx_model(
    model='model.simplify.onnx', # Simplified using onnxsim web tool
    output_dir='hls4ml_prj',
    project_name='project',
    backend='Vitis',
    hls_config=config,
    part='xcvu13p-fsga2577-2-e',
    clock_period=5,
    io_type='io_parallel'
)

{'Model': {'Precision': 'ap_fixed<16,6>',
           'ReuseFactor': 1,
           'Strategy': 'Latency'}}

Interpreting Model ...
Output layers:  ['/projector/projector.1/Relu']
Input shape: [None, 400, 14]
Topology:
Layer name: /input_proj/MatMul, layer type: Dense, current shape: [[None, 400, 14]]
Layer name: /input_proj/Add, layer type: BiasAdd, current shape: [[None, 400, 128]]
Layer name: /encoder/q_proj/MatMul, layer type: Dense, current shape: [[None, 400, 128]]
Layer name: /encoder/q_proj/Add, layer type: BiasAdd, current shape: [[None, 400, 128]]
Layer name: /encoder/k_proj/MatMul, layer type: Dense, current shape: [[None, 400, 128]]
Layer name: /encoder/k_proj/Add, layer type: BiasAdd, current shape: [[None, 400, 128]]
Layer name: /encoder/v_proj/MatMul, layer type: Dense, current shape: [[None, 400, 128]]
Layer name: /encoder/v_proj/Add, layer type: BiasAdd, current shape: [[None, 400, 128]]
Layer name: /encoder/attn/MatMul, layer type: Dense, current shape: [[None, 400, 128

UnboundLocalError: local variable 'data' referenced before assignment

In [44]:
hls4ml.converters.onnx_to_hls??

Signature: hls4ml.converters.onnx_to_hls(config)
Source:   
@requires('onnx')
def onnx_to_hls(config):
    """Convert onnx model to hls model from configuration.

    Args:
        config (dict): ONNX configuration from yaml file or passed through API.

    Raises:
        Exception: Raised if an unsupported operation is found in the ONNX model.

    Returns:
        ModelGraph: hls4ml model object
    """

    # Extract model architecture
    print('Interpreting Model ...')

    import onnx

    onnx_model = onnx.load(config['OnnxModel']) if isinstance(config['OnnxModel'], str) else config['OnnxModel']

    layer_list, input_layers, output_layers = parse_onnx_model(onnx_model)

    #################
    # Generate HLS
    #################

    print('Creating HLS model')
    hls_model = ModelGraph.from_layer_list(config, layer_list, input_layers, output_layers)
    return hls_model
File:      /scratch/rcruzcan/TAC-HEP-FPGA-Project/venv13/lib64/python3.13/site-packages/hls4ml/converte